# 02 Simulate SNe Ia in the LSST DDFs with lightcurvelynx

- author : Sylvie Dagoret-Campagne
- creation date : 2026-10-30
- based on : `../Example_notebooks_madeleine2026/Simulate_DDF.ipynb` (Madeleine Ginolin, `skysurvey`), rewritten with `lightcurvelynx`
- same implementation style as `01_SimulateSNLightCurvelynx.ipynb` (population graph -> `simulate_lightcurves` -> emcee posteriors)
- kernel : `lightcurvelynx_py313`
- python dependances : The kernel `lightcurvelynx_py313`  : Do conda install conda-forge::dust_extinction

## Overview

Same steps as the `skysurvey` notebook, with the same inputs (per-CCD visit table of the six DDFs, SNANA search efficiency,
Rubin bandpasses, SNANA-like SN Ia population), but all the simulation machinery is `lightcurvelynx`:

1. **Inputs**: per-CCD visit table (`edp2_detectorvisit_table.parquet`), converted to the column names and units expected by `lightcurvelynx` (zero point in nJy per electron, sky variance in e-^2).
2. **Counting**: expected number of SNe Ia per DDF from the Frohmaier+2019 rate, the cosmology, the field area and the duration of the visits.
3. **Simulation**: for each DDF, a Poisson number of SNe Ia is drawn (0.05 < z < 1.2) and observed with `simulate_lightcurves`
   (one `LSSTObsTable` per field with the CCD footprint; SALT3 + Tripp; asymmetric-Gaussian x1 and c; F99 Milky Way dust from SFD).
4. **Processing**: nightly co-addition (inverse-variance), SNANA search efficiency, then >= 5 sigma detections in >= 3 bands.
5. **Fits**: Bayesian posteriors (`emcee`) of (t0, mB, x1, c) with the redshift fixed to its true value, and optionally with a free redshift.
6. **Selection, counting and validation**: SALT quality cuts, number of SNe per field at each stage, pulls (fit - truth)/error and an example light curve.

All fluxes are in nJy (the native unit of `lightcurvelynx`), so the common ZP = 30 step of the original notebook is not needed.

## Imports and configuration

In [ ]:
import os
import pickle
import time
import warnings

import emcee
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from astropy.cosmology import FlatLambdaCDM
from scipy.integrate import trapezoid
from scipy.interpolate import CubicSpline
from scipy.stats import chi2 as chi2_dist
from scipy.stats import norm
from tqdm.auto import tqdm

from lightcurvelynx.astro_utils.dustmap import ConstantHemisphereDustMap, DustmapWrapper
from lightcurvelynx.astro_utils.passbands import Passband, PassbandGroup
from lightcurvelynx.astro_utils.snia_utils import DistModFromRedshift, X0FromDistMod
from lightcurvelynx.effects.extinction import ExtinctionEffect
from lightcurvelynx.graph_state import GraphState
from lightcurvelynx.math_nodes.given_sampler import GivenValueSampler
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.models.sncosmo_models import SncosmoWrapperModel
from lightcurvelynx.obstable.lsst_obstable import LSSTObsTable
from lightcurvelynx.simulate import simulate_lightcurves
from lightcurvelynx.survey_info import SurveyInfo
from lightcurvelynx.utils.extrapolate import LinearDecay, ZeroPadding

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
NB_TAG = "02_SimSNIa_DDF"
DATA_DIR = f"data_{NB_TAG}"
FIG_DIR = f"figs_{NB_TAG}"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ---- input files (from the skysurvey notebook of Madeleine Ginolin) ----
MADELEINE_DIR = os.path.join("..", "Example_notebooks_madeleine2026")
VISITS_FILE = os.path.join(MADELEINE_DIR, "edp2_detectorvisit_table.parquet")
SEARCHEFF_FILE = os.path.join(MADELEINE_DIR, "SEARCHEFF_PIPELINE_LSST_DC2_Run2.2i_WFD.DAT")
BANDS = ["u", "g", "r", "i", "z", "y"]
FILTER_FILES = {b: os.path.join(MADELEINE_DIR, f"filter_lsst_{b}.npy") for b in BANDS}
DDF_FIELDS = ["COSMOS", "ECDFS", "EDFS_a", "EDFS_b", "ELAISS1", "XMM_LSS"]

# ---- SN Ia population (same values as the skysurvey notebook) ----
SALT_MODEL = "salt3"
H0, OMEGA_M = 70.0, 0.315  # cosmology agreed at the DESC July 2026 meeting
ALPHA, BETA = 0.15, 3.15  # Tripp: skysurvey alpha = -0.15 is the same relation as +0.15 in lightcurvelynx
M_ABS_MEAN, M_ABS_SIGMA = -19.3, 0.1
X1_PARS = dict(mu=0.973, sigma_left=1.472, sigma_right=0.222)  # asymmetric Gaussian (SNANA)
C_PARS = dict(mu=-0.054, sigma_left=0.043, sigma_right=0.101)
Z_MIN, Z_MAX = 0.05, 1.2
PHASE_WINDOW = (-20.0, 40.0)  # rest-frame phase range of the simulated light curves (days)
MB0 = 10.635  # lightcurvelynx / sncosmo convention: mB = -2.5 log10(x0) + MB0
R0_FROHMAIER, ALPHA_FROHMAIER = 2.3e4, 1.70  # rate r0 (1+z)^alpha in Gpc^-3 yr^-1
SIDE_CCD = np.sqrt(
    9.6 / 189
)  # side (deg) of a square CCD footprint, used for the draw box of the SN positions
SIM_FRACTION = 1.0  # fraction of the expected number of SNe that is simulated (reduce for quick tests)
SEED = 42

# ---- selection (same as the skysurvey notebook) ----
NSIGMA_DET = 5.0
NBANDS_MIN = 3

# ---- Milky Way dust ----
USE_MW_DUST = True
if "nersc" in os.uname().nodename.lower():
    DUST_DATA_DIR = os.path.join(os.getcwd(), "ASTROMAPS")
else:
    DUST_DATA_DIR = os.path.join(os.path.expanduser("~"), "DATA", "ASTROMAPS")

# ---- Bayesian fits ----
FIT_MODES = ["fixed_z"]  # add "free_z" to also fit the redshift
N_FIT = 20  # maximum number of detected SNe fitted per mode (each fit takes ~0.5-1 min)
N_WALKERS, N_STEPS, N_BURN = 16, 700, 250
PRIOR_MODE = "flat"  # "flat" within the bounds below (like sncosmo.fit_lc) or "population"
PRIOR_BOUNDS = {"x1": (-10.0, 10.0), "c": (-1.0, 3.0), "mB": (15.0, 32.0), "z": (0.01, 1.5)}
CUT_MWEBV = 0.3


def save_fig(fig, name):
    """Save a figure as both PDF and PNG in FIG_DIR."""
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(FIG_DIR, f"{NB_TAG}_{name}.{ext}"), dpi=150, bbox_inches="tight")

## DustMap path

In [ ]:
path_rubin_sim_data_path_name = "RUBIN_SIM_DATA_DIR"
path_rubin_sim_data_path = os.getenv(path_rubin_sim_data_path_name)

In [ ]:
path_rubin_sim_data_path

In [ ]:
dustmap_path_name = "DUST_DATA_DIR"
path_dustmap_path = os.getenv(dustmap_path_name)
DUST_DATA_DIR = path_dustmap_path

In [ ]:
path_dustmap_path

## Per-CCD visit table of the DDFs

One row per visit and detector (`field`, `band`, `ra`, `dec`, `mjd`, ...), prepared by Madeleine for `skysurvey`. Its columns are not assumed:
we print the schema, then detect which noise description the table provides.

- **`skysurvey` scheme** (the table of Madeleine): `zp` (AB zero point in counts), `skynoise` (rms of the sky noise in counts, integrated over the measurement),
  `gain` (e-/count). The flux error is `sqrt(skynoise^2 + flux/gain)` in counts. It is reproduced exactly with the Poisson noise model of `lightcurvelynx`
  by giving it: `zp` = flux of one electron in nJy = `10^(-0.4 (zp - 31.4)) / gain`; `sky_bg_e` = `(skynoise * gain)^2` (sky variance in e-^2) with a PSF footprint of 1 pixel;
  read noise = dark current = 0. The `filter` column of the table (physical filter) is not used: the band is taken from `band`.
- **DP2 scheme** (`VisitDetector` table): `zeroPoint`, `seeing`, `skyBg`, ... as in the standard `from_ccdvisit_table` loader.

In [ ]:
pf = pq.ParquetFile(VISITS_FILE)
all_cols = pf.schema_arrow.names
print(f"{pf.metadata.num_rows} rows, {len(all_cols)} columns")
print(all_cols)

In [ ]:
def first_of(*names):
    """First of the given column names that exists in the table (or None)."""
    return next((n for n in names if n in all_cols), None)


col = {
    "field": first_of("field"),
    "band": first_of("band"),
    "ra": first_of("ra"),
    "dec": first_of("dec"),
    "time": first_of("mjd", "expMidptMJD", "obsStartMJD"),
    "zp": first_of("zp", "zeroPoint"),
    "skynoise": first_of("skynoise", "skyNoise"),
    "gain": first_of("gain"),
    "maglim": first_of("magLim", "maglim"),
    "seeing": first_of("seeing"),
    "skybg": first_of("skyBg"),
    "exptime": first_of("expTime", "exptime"),
    "pixscale": first_of("pixelScale"),
}
core = ["field", "band", "ra", "dec", "time", "zp"]
missing = [k for k in core if col[k] is None]
if missing:
    raise ValueError(f"Missing required columns {missing}. Columns: {all_cols}")
if col["skynoise"] and col["gain"]:
    NOISE_SCHEME = "skysurvey"
elif col["seeing"] and col["skybg"]:
    NOISE_SCHEME = "dp2"
else:
    raise ValueError(f"Need (skynoise, gain) or (seeing, skyBg) to describe the noise. Columns: {all_cols}")
print("noise scheme:", NOISE_SCHEME)
print("columns used:", {k: v for k, v in col.items() if v})

raw = pd.read_parquet(VISITS_FILE, columns=sorted({v for v in col.values() if v}))


def numeric(name):
    return pd.to_numeric(raw[name], errors="coerce")


visits = pd.DataFrame(
    {
        "field": raw[col["field"]].astype(str),
        # normalise the band names (lsstu / lsst_u / rubinu / u -> u)
        "band": raw[col["band"]]
        .astype(str)
        .str.replace(r"^(lsst|rubin)_?", "", regex=True, case=False)
        .str.lower(),
        "ra": numeric(col["ra"]),
        "dec": numeric(col["dec"]),
        "expMidptMJD": numeric(col["time"]),
        "expTime": numeric(col["exptime"]) if col["exptime"] else 30.0,
    }
)
if col["maglim"]:
    visits["magLim"] = numeric(col["maglim"])

if NOISE_SCHEME == "skysurvey":
    gain = numeric(col["gain"])
    njy_per_count = 10.0 ** (-0.4 * (numeric(col["zp"]) - 31.4))  # flux of one count (nJy), AB system
    visits["zp"] = njy_per_count / gain  # flux of one electron (nJy), as expected by lightcurvelynx
    visits["sky_bg_e"] = (
        numeric(col["skynoise"]) * gain
    ) ** 2  # sky variance (e-^2) in the measurement aperture
    visits["psf_footprint"] = 1.0  # the aperture is already included in skynoise
    must_be_positive = ["zp", "sky_bg_e"]
    OBS_KWARGS = dict(
        read_noise=0.0, dark_current=0.0, zp_err_mag=0.0
    )  # all the noise is in skynoise and gain
else:
    visits["zeroPoint"] = numeric(col["zp"])
    visits["seeing"] = numeric(col["seeing"])
    visits["skyBg"] = numeric(col["skybg"])
    visits["pixelScale"] = numeric(col["pixscale"]) if col["pixscale"] else 0.2
    must_be_positive = ["skyBg", "seeing"]
    OBS_KWARGS = {}

visits = visits[visits["band"].isin(BANDS)]

# drop rows that lightcurvelynx would reject (non-finite or non-physical noise information)
num_cols = [c for c in visits.columns if c not in ("field", "band")]
good = np.isfinite(visits[num_cols]).all(axis=1) & (visits[must_be_positive] > 0).all(axis=1)
print(f"{(~good).sum()} rows dropped (non-finite or non-physical values) out of {len(visits)}")
visits = visits[good].reset_index(drop=True)

print("rows per field:", visits["field"].value_counts().to_dict())
unknown = [f for f in DDF_FIELDS if f not in set(visits["field"])]
if unknown:
    print("WARNING: fields missing in the table:", unknown)
visits.head()

## Rubin bandpasses

The transmission curves are the ones of the `skysurvey` notebook (`filter_lsst_<band>.npy`, from the butler; wavelengths in nm).
They are converted to `lightcurvelynx` passbands; if the files are missing the LSST baseline throughputs are downloaded instead.

In [ ]:
passbands, band_tables = [], {}
for b in BANDS:
    if os.path.exists(FILTER_FILES[b]):
        wave_nm, trans = np.load(FILTER_FILES[b])
        band_tables[b] = (wave_nm, trans / trans.max())
        passbands.append(Passband(np.column_stack(band_tables[b]), "LSST", b, units="nm"))
if len(passbands) == len(BANDS):
    passband_group = PassbandGroup(given_passbands=passbands)
    print("Bandpasses read from", MADELEINE_DIR)
else:
    band_tables = {}
    passband_group = PassbandGroup.from_preset(preset="LSST", table_dir=os.path.join(DATA_DIR, "passbands"))
    print("Bandpass files not found: LSST baseline throughputs used")

fig, ax = plt.subplots(figsize=(7, 3.5))
for b, (wave_nm, trans) in band_tables.items():
    ax.plot(wave_nm, trans, label=b)
ax.set_xlabel("wavelength (nm)")
ax.set_ylabel("normalised transmission")
ax.legend()
save_fig(fig, "bandpasses")

## Milky Way dust

E(B-V) is read from the SFD maps with `dustmaps` (same maps and directory convention as in the `skysurvey` notebook) and
applied with the F99 law in the observer frame. If the maps are not found, a constant E(B-V) = 0.02 is used instead.

In [ ]:
def get_dust_map():
    """Return (dust map object with a query(SkyCoord) method, description)."""
    if USE_MW_DUST:
        sfd_dir = os.path.join(DUST_DATA_DIR, "sfd")
        files = ["SFD_dust_4096_ngp.fits", "SFD_dust_4096_sgp.fits"]
        if all(os.path.exists(os.path.join(sfd_dir, f)) for f in files):
            from dustmaps.sfd import SFDQuery

            return SFDQuery(map_dir=sfd_dir), f"SFD maps from {sfd_dir}"
        print(f"WARNING: SFD maps not found in {sfd_dir}")
    return ConstantHemisphereDustMap(0.02, 0.02), "constant E(B-V) = 0.02"


dust_map, dust_description = get_dust_map()
print("Milky Way dust:", dust_description)

## Population model and expected number of SNe Ia

- volumetric rate: Frohmaier+2019, `r(z) = r0 (1+z)^alpha`; redshift density `r(z)/(1+z) dV/dz`
- x1 and c: asymmetric Gaussians (SNANA); magnitude from the Tripp relation with Gaussian intrinsic scatter

In [ ]:
cosmo = FlatLambdaCDM(H0=H0, Om0=OMEGA_M)


def asymmetric_gaussian(xx, mu, sigma_left, sigma_right):
    """Asymmetric Gaussian density on the grid xx (SNANA parametrisation), normalised to 1."""
    sigma = np.where(xx < mu, sigma_left, sigma_right)
    norm_factor = np.sqrt(2 / np.pi) / (sigma_left + sigma_right)
    return norm_factor * np.exp(-0.5 * ((xx - mu) / sigma) ** 2)


def frohmaier_2019(z, r0=R0_FROHMAIER, alpha=ALPHA_FROHMAIER):
    """Volumetric SN Ia rate in Gpc^-3 yr^-1."""
    return r0 * (1 + z) ** alpha


def redshift_density(z):
    """Number of SNe per unit redshift, per year (observer frame) and per steradian."""
    dvdz = cosmo.differential_comoving_volume(z).value * 1e-9  # Gpc^3 per sr per unit z
    return frohmaier_2019(z) / (1 + z) * dvdz


def expected_sn_count(area_deg2, duration_days, z_min=Z_MIN, z_max=Z_MAX):
    """Expected number of SNe Ia for a given sky area and observer-frame duration."""
    z_fine = np.linspace(z_min, z_max, 2000)
    per_year_per_sr = trapezoid(redshift_density(z_fine), z_fine)
    area_sr = area_deg2 * (np.pi / 180.0) ** 2
    return per_year_per_sr * area_sr * duration_days / 365.25


# discrete grids used to draw z, x1 and c with the requested densities
Z_GRID = np.linspace(Z_MIN, Z_MAX, 4000)
Z_WEIGHTS = redshift_density(Z_GRID)
X1_GRID = np.arange(-4, 4, 0.005)
X1_WEIGHTS = asymmetric_gaussian(X1_GRID, **X1_PARS)
C_GRID = np.arange(-0.3, 1.0, 0.001)
C_WEIGHTS = asymmetric_gaussian(C_GRID, **C_PARS)

### Fields: footprint boxes, durations and expected counts

As in the `skysurvey` notebook, the SNe are drawn in the bounding box of the pointings (+ one CCD side), with peak dates between
30 days before the first visit and 30 days after the last one. Only SNe falling on a CCD at an observed date give light-curve points.

In [ ]:
rows = []
for field in DDF_FIELDS:
    lg = visits[visits["field"] == field]
    if len(lg) == 0:
        continue
    ra_range = (lg["ra"].min() - SIDE_CCD, lg["ra"].max() + SIDE_CCD)
    dec_range = (lg["dec"].min() - SIDE_CCD, lg["dec"].max() + SIDE_CCD)
    dec_mid = np.deg2rad(0.5 * (dec_range[0] + dec_range[1]))
    area = (ra_range[1] - ra_range[0]) * np.cos(dec_mid) * (dec_range[1] - dec_range[0])
    t_start, t_stop = lg["expMidptMJD"].min() - 30, lg["expMidptMJD"].max() + 30
    rows.append(
        {
            "field": field,
            "n_ccd_rows": len(lg),
            "ra_min": ra_range[0],
            "ra_max": ra_range[1],
            "dec_min": dec_range[0],
            "dec_max": dec_range[1],
            "area_deg2": area,
            "t_start": t_start,
            "t_stop": t_stop,
            "n_expected": expected_sn_count(area, t_stop - t_start),
        }
    )
field_info = pd.DataFrame(rows).set_index("field")
field_info.round(2)

## SN Ia source model (one per field)

In [ ]:
def build_source(info):
    """Build the SALT3 SN Ia source of a field: population graph + F99 Milky Way extinction."""
    z_func = GivenValueSampler(Z_GRID, weights=Z_WEIGHTS)
    x1_func = GivenValueSampler(X1_GRID, weights=X1_WEIGHTS)
    c_func = GivenValueSampler(C_GRID, weights=C_WEIGHTS)
    m_abs_func = NumpyRandomFunc("normal", loc=M_ABS_MEAN, scale=M_ABS_SIGMA)
    distmod_func = DistModFromRedshift(z_func, H0=H0, Omega_m=OMEGA_M)
    x0_func = X0FromDistMod(
        distmod=distmod_func, x1=x1_func, c=c_func, alpha=ALPHA, beta=BETA, m_abs=m_abs_func
    )
    t0_func = NumpyRandomFunc("uniform", low=info["t_start"], high=info["t_stop"])
    ra_func = NumpyRandomFunc("uniform", low=info["ra_min"], high=info["ra_max"])
    dec_func = NumpyRandomFunc("uniform", low=info["dec_min"], high=info["dec_max"])
    ebv_node = DustmapWrapper(dust_map, ra=ra_func, dec=dec_func, node_label="dust")

    source = SncosmoWrapperModel(
        SALT_MODEL,
        t0=t0_func,
        x0=x0_func,
        x1=x1_func,
        c=c_func,
        ra=ra_func,
        dec=dec_func,
        redshift=z_func,
        node_label="source",
        time_extrapolation=LinearDecay(50.0),
        wave_extrapolation=ZeroPadding(),
    )
    source.add_effect(
        ExtinctionEffect(
            extinction_model="F99", ebv=ebv_node, frame="observer", r_v=3.1, backend="dust_extinction"
        )
    )
    return source

## Simulation, field by field

For each DDF: build the `LSSTObsTable` from the per-CCD rows (with the CCD detector footprint), draw a Poisson number of SNe around the
expected count, and observe them. The SNe without any observation are dropped by `simulate_lightcurves`.

In [ ]:
sources, sim_rows, lc_tables, params_store = {}, [], [], {}
n_drawn = {}
rng_master = np.random.default_rng(SEED)
t_begin = time.time()

for k, field in enumerate(field_info.index):
    info = field_info.loc[field]
    logs = visits[visits["field"] == field].drop(columns=["field"])
    obs_table = LSSTObsTable.from_ccdvisit_table(logs, **OBS_KWARGS)
    survey_info = SurveyInfo(obstable=obs_table, passbands=passband_group, survey_name="LSST")
    source = build_source(info)
    sources[field] = source

    n_sn = int(rng_master.poisson(info["n_expected"] * SIM_FRACTION))
    n_drawn[field] = n_sn
    if n_sn == 0:
        continue
    lightcurves = simulate_lightcurves(
        source,
        n_sn,
        survey_info,
        rest_time_window_offset=PHASE_WINDOW,
        rng=np.random.default_rng(SEED + 1 + k),
        drop_empty_rows=True,
        progress_bar=False,
    )
    for i in range(len(lightcurves)):
        p = dict(lightcurves.iloc[i]["params"])
        sn = f"{field}_{i}"
        params_store[sn] = p
        sim_rows.append(
            {
                "sn": sn,
                "field": field,
                "z": p["source.redshift"],
                "t0": p["source.t0"],
                "x0": p["source.x0"],
                "x1": p["source.x1"],
                "c": p["source.c"],
                "mwebv": p["source.ebv"],
                "ra": p["source.ra"],
                "dec": p["source.dec"],
            }
        )
        df = pd.DataFrame(lightcurves.iloc[i]["lightcurve"])[["mjd", "filter", "flux", "fluxerr"]]
        df["sn"] = sn
        lc_tables.append(df)
    print(f"{field}: expected {info['n_expected']:.0f}, drawn {n_sn}, observed {len(lightcurves)}")

sim_params = pd.DataFrame(sim_rows)
lcs = pd.concat(lc_tables, ignore_index=True)
sim_params["mb"] = -2.5 * np.log10(sim_params["x0"]) + MB0
print(
    f"Total: {len(sim_params)} SNe with observations, {len(lcs)} light-curve points ({time.time() - t_begin:.0f} s)"
)
sim_params.to_parquet(os.path.join(DATA_DIR, f"{NB_TAG}_sim_params.parquet"))

## Nightly co-addition and search efficiency

Points of the same SN, band and night are co-added with inverse-variance weights. The SNANA search efficiency (probability that a point of a
given S/N passes the photometry pipeline, estimated by Sanchez+22) is then applied band by band.

In [ ]:
good_pts = np.isfinite(lcs["flux"]) & np.isfinite(lcs["fluxerr"]) & (lcs["fluxerr"] > 0)
lcs = lcs[good_pts].copy()
lcs["night"] = np.floor(lcs["mjd"]).astype(int)
lcs["w"] = 1.0 / lcs["fluxerr"] ** 2
lcs["wf"] = lcs["flux"] * lcs["w"]
lcs["wt"] = lcs["mjd"] * lcs["w"]
grp = lcs.groupby(["sn", "filter", "night"], sort=False).agg(
    w=("w", "sum"), wf=("wf", "sum"), wt=("wt", "sum"), n=("w", "size")
)
lcs_coadd = grp.reset_index()
lcs_coadd["flux"] = lcs_coadd["wf"] / lcs_coadd["w"]
lcs_coadd["fluxerr"] = 1.0 / np.sqrt(lcs_coadd["w"])
lcs_coadd["time"] = lcs_coadd["wt"] / lcs_coadd["w"]
lcs_coadd["snr"] = lcs_coadd["flux"] / lcs_coadd["fluxerr"]
lcs_coadd = lcs_coadd.drop(columns=["w", "wf", "wt"])
print(f"{len(lcs)} points -> {len(lcs_coadd)} nightly co-added points")

In [ ]:
# search efficiency curves: the file holds the 6 bands one after the other, in blocks of 101 lines
snr_grid = np.loadtxt(SEARCHEFF_FILE, skiprows=27, max_rows=99, usecols=[1])
searcheff_funcs = {}
for i, b in enumerate(BANDS):
    eff = np.loadtxt(SEARCHEFF_FILE, skiprows=27 + i * 101, max_rows=99, usecols=[2])
    searcheff_funcs[b] = CubicSpline(snr_grid, eff)

eff = np.zeros(len(lcs_coadd))
for b in BANDS:
    m = (lcs_coadd["filter"] == b).to_numpy()
    if m.any():
        eff[m] = searcheff_funcs[b](lcs_coadd.loc[m, "snr"])
eff = np.clip(np.nan_to_num(eff, nan=0.0), 0.0, 1.0)
keep = np.random.default_rng(SEED + 100).binomial(1, eff).astype(bool)
lcs_coadd["search_eff"] = eff
lcs_sel = lcs_coadd[keep].reset_index(drop=True)
print(f"search efficiency: {len(lcs_sel)} / {len(lcs_coadd)} co-added points kept")

## Light-curve quality cuts and counting

A point is a detection if S/N >= `NSIGMA_DET`. With the rest-frame phase `(t - t0)/(1+z)`, we count the detections between -10 and +40 days
in each band, and keep the SNe detected in at least `NBANDS_MIN` bands (the cut is made on the truth t0 and z, as in the original notebook).

In [ ]:
merged = lcs_sel.merge(sim_params[["sn", "t0", "z"]], on="sn")
merged["phase"] = (merged["time"] - merged["t0"]) / (1 + merged["z"])
det = merged[(merged["snr"] >= NSIGMA_DET) & merged["phase"].between(-10, 40)]
ndet_band = det.groupby(["sn", "filter"]).size()
nbands = ndet_band.groupby(level=0).size()
detected_sn = nbands[nbands >= NBANDS_MIN].index
sim_params["detected"] = sim_params["sn"].isin(detected_sn)
print(
    f"{len(detected_sn)} SNe detected at >= {NSIGMA_DET} sigma in >= {NBANDS_MIN} bands out of {len(sim_params)} observed"
)

counts = (
    pd.DataFrame(
        {
            "expected": field_info["n_expected"] * SIM_FRACTION,
            "drawn": pd.Series(n_drawn),
            "observed": sim_params.groupby("field").size(),
            "detected": sim_params[sim_params["detected"]].groupby("field").size(),
        }
    )
    .reindex(field_info.index)
    .fillna(0)
)
counts.round(1)

## Bayesian fits of the SALT3 parameters

For each detected SN, the posterior of $(t_0, m_B, x_1, c)$ (and $z$ in the `free_z` mode) is sampled with `emcee` from the co-added points.
The likelihood is Gaussian with the co-added flux errors and uses the same `lightcurvelynx` model (SALT3 + F99 dust) as the simulation;
$E(B-V)$ is fixed to its true value, as in the original fits. The walkers start around the true values (the original notebook also starts
the fits from the true z and from t0 + N(0, 10 d)).

In [ ]:
pts_by_sn = {sn: g for sn, g in lcs_sel.groupby("sn")}
field_of = sim_params.set_index("sn")["field"].to_dict()


def log_asym_gauss(x, mu, sigma_left, sigma_right):
    sigma = sigma_left if x < mu else sigma_right
    return -0.5 * ((x - mu) / sigma) ** 2


def theta_to_params(base_params, theta, free_z):
    """Replace the SALT parameters of a simulation parameter dictionary by theta."""
    d = dict(base_params)
    d["source.t0"] = theta[0]
    d["source.x0"] = 10.0 ** (-0.4 * (theta[1] - MB0))
    d["source.x1"] = theta[2]
    d["source.c"] = theta[3]
    if free_z:
        d["source.redshift"] = theta[4]
    return d


def fit_sn(sn, free_z, rng, n_walkers=N_WALKERS, n_steps=N_STEPS, n_burn=N_BURN):
    """Sample the posterior of one SN with emcee."""
    pts = pts_by_sn[sn]
    source = sources[field_of[sn]]
    base = params_store[sn]
    t = pts["time"].to_numpy(float)
    filt = pts["filter"].to_numpy(str)
    flux = pts["flux"].to_numpy(float)
    err = pts["fluxerr"].to_numpy(float)
    t_lo, t_hi = t.min() - 30.0, t.max() + 30.0
    names = ["t0", "mB", "x1", "c"] + (["z"] if free_z else [])

    def model_flux(theta):
        return source.evaluate_bandfluxes(
            passband_group, t, filt, GraphState.from_dict(theta_to_params(base, theta, free_z))
        )

    def log_prob(theta):
        t0, mB, x1, c = theta[:4]
        if not (t_lo < t0 < t_hi):
            return -np.inf
        for val, key in ((mB, "mB"), (x1, "x1"), (c, "c")):
            if not (PRIOR_BOUNDS[key][0] < val < PRIOR_BOUNDS[key][1]):
                return -np.inf
        if free_z and not (PRIOR_BOUNDS["z"][0] < theta[4] < PRIOR_BOUNDS["z"][1]):
            return -np.inf
        lp = 0.0
        if PRIOR_MODE == "population":
            lp = log_asym_gauss(x1, **X1_PARS) + log_asym_gauss(c, **C_PARS)
        m = model_flux(theta)
        if not np.all(np.isfinite(m)):
            return -np.inf
        return lp - 0.5 * np.sum(((flux - m) / err) ** 2)

    truth = np.array(
        [base["source.t0"], -2.5 * np.log10(base["source.x0"]) + MB0, base["source.x1"], base["source.c"]]
    )
    spread = np.array([3.0, 0.1, 0.3, 0.03])
    if free_z:
        truth = np.append(truth, base["source.redshift"])
        spread = np.append(spread, 0.01)
    p0 = truth + rng.normal(size=(n_walkers, len(truth))) * spread
    sampler = emcee.EnsembleSampler(n_walkers, len(truth), log_prob)
    sampler.run_mcmc(p0, n_steps, progress=False)
    chain = sampler.get_chain(discard=n_burn, flat=True)
    logp = sampler.get_log_prob(discard=n_burn, flat=True)
    theta_map = chain[np.argmax(logp)]
    chisq_map = float(np.sum(((flux - model_flux(theta_map)) / err) ** 2))
    return {
        "sn": sn,
        "free_z": free_z,
        "names": names,
        "truth": truth,
        "chain": chain[:: max(1, len(chain) // 2000)],
        "mean": chain.mean(axis=0),
        "std": chain.std(axis=0),
        "theta_map": theta_map,
        "chisq": chisq_map,
        "ndof": len(t) - len(truth),
        "tau": sampler.get_autocorr_time(tol=0),
        "acceptance": sampler.acceptance_fraction.mean(),
    }

In [ ]:
fit_file = os.path.join(DATA_DIR, f"{NB_TAG}_posteriors.pkl")
fit_results = pickle.load(open(fit_file, "rb")) if os.path.exists(fit_file) else {}
fit_rng = np.random.default_rng(SEED + 7)
sn_to_fit = list(detected_sn)[:N_FIT]

for mode in FIT_MODES:
    free_z = mode == "free_z"
    for sn in tqdm(sn_to_fit, desc=mode):
        if (mode, sn) in fit_results:
            continue
        fit_results[(mode, sn)] = fit_sn(sn, free_z, fit_rng)
        with open(fit_file, "wb") as f:
            pickle.dump(fit_results, f)
print(f"{len(fit_results)} posteriors available")

## Fit summary, SALT quality cuts and counting

Quality cuts of the original notebook: `-0.3 < c < 0.3` and `sigma_c < 0.2`; `-3 < x1 < 3` and `sigma_x1 < 1`; `sigma_t0 < 2` d;
fit probability > 1e-4 (chi2 at the maximum-posterior sample); E(B-V) < 0.3.

In [ ]:
def summarize_fits(mode):
    rows = []
    for (m, sn), r in fit_results.items():
        if m != mode:
            continue
        d = dict(zip(r["names"], r["mean"]))
        s = dict(zip(r["names"], r["std"]))
        tr = dict(zip(r["names"], r["truth"]))
        row = {
            "sn": sn,
            "field": field_of[sn],
            "mwebv": params_store[sn]["source.ebv"],
            "chisq": r["chisq"],
            "ndof": r["ndof"],
            "tau_max": r["tau"].max(),
            "acc": r["acceptance"],
        }
        for k in r["names"]:
            row[k], row[f"{k}_err"], row[f"{k}_true"] = d[k], s[k], tr[k]
            row[f"{k}_pull"] = (d[k] - tr[k]) / s[k]
        if "z" not in r["names"]:
            row["z"], row["z_true"] = params_store[sn]["source.redshift"], params_store[sn]["source.redshift"]
        rows.append(row)
    df = pd.DataFrame(rows)
    if len(df):
        df["fitprob"] = chi2_dist.sf(df["chisq"], df["ndof"].clip(lower=1))
    return df


def apply_cuts(df):
    mask_c = df["c"].between(-0.3, 0.3) & (df["c_err"] < 0.2)
    mask_x1 = df["x1"].between(-3, 3) & (df["x1_err"] < 1)
    mask_fit = (df["t0_err"] < 2) & (df["fitprob"] > 1e-4) & (df["mwebv"] < CUT_MWEBV)
    return df[mask_c & mask_x1 & mask_fit]


fit_tables, selected_tables = {}, {}
for mode in FIT_MODES:
    fit_tables[mode] = summarize_fits(mode)
    selected_tables[mode] = apply_cuts(fit_tables[mode]) if len(fit_tables[mode]) else fit_tables[mode]
    print(f"[{mode}] {len(fit_tables[mode])} fitted, {len(selected_tables[mode])} pass the SALT cuts")

mode_main = FIT_MODES[0]
fit_tables[mode_main][
    ["sn", "field", "z", "x1", "x1_err", "c", "c_err", "t0_err", "fitprob", "tau_max", "acc"]
].round(3).head(10)

In [ ]:
# counting per field at each stage
counts["fitted"] = (
    fit_tables[mode_main].groupby("field").size().reindex(field_info.index).fillna(0)
    if len(fit_tables[mode_main])
    else 0
)
counts["selected"] = (
    selected_tables[mode_main].groupby("field").size().reindex(field_info.index).fillna(0)
    if len(selected_tables[mode_main])
    else 0
)
counts.to_csv(os.path.join(DATA_DIR, f"{NB_TAG}_counts_per_field.csv"))
print(f"Number of SNe Ia per DDF (fits limited to the first {N_FIT} detected SNe overall)")
display(counts.round(1))

fig, ax = plt.subplots(figsize=(9, 4))
width = 0.13
for j, col in enumerate(["expected", "drawn", "observed", "detected", "fitted", "selected"]):
    ax.bar(np.arange(len(counts)) + (j - 2.5) * width, counts[col], width, label=col)
ax.set_xticks(np.arange(len(counts)))
ax.set_xticklabels(counts.index)
ax.set_ylabel("number of SNe Ia")
ax.legend(ncol=3, frameon=False)
save_fig(fig, "counts_per_field")

## Validation: pulls

In [ ]:
nbins = 30
bins_pull = np.linspace(-5, 5, nbins)
xpull = np.linspace(-5, 5, 200)

for mode in FIT_MODES:
    sel = selected_tables[mode]
    if len(sel) < 2:
        print(f"[{mode}] not enough selected SNe for the pull plots ({len(sel)})")
        continue
    fig, ax = plt.subplots(4, 3, figsize=(14, 8))
    ax[0, 0].hist(sel["x1_true"], bins=np.linspace(-3, 3, nbins), color="teal", alpha=0.3)
    ax[0, 0].set_xlabel(r"True $x_1$")
    ax[0, 0].set_ylabel("Number of SNe")
    ax[0, 1].hist(sel["c_true"], bins=np.linspace(-0.3, 0.8, nbins), color="teal", alpha=0.3)
    ax[0, 1].set_xlabel(r"True $c$")
    ax[0, 2].hist(sel["t0_true"], bins=nbins, color="teal", alpha=0.3)
    ax[0, 2].set_xlabel(r"True $t_0$")
    for a, key, label in (
        (ax[1, 0], "x1", r"$x_1$"),
        (ax[1, 1], "c", r"$c$"),
        (ax[1, 2], "t0", r"$t_0$"),
        (ax[3, 0], "mB", r"$m_B$"),
    ):
        a.hist(sel[f"{key}_pull"], bins=bins_pull, color="darkred", alpha=0.5, density=True)
        a.plot(xpull, norm.pdf(xpull), color="black", alpha=0.5, label=r"$\mathcal{N}(0,1)$")
        a.set_xlabel(f"Pull on {label}")
        a.set_title(f"mean={sel[f'{key}_pull'].mean():.2f}, std={sel[f'{key}_pull'].std():.2f}", fontsize=9)
    ax[1, 0].legend()
    ax[1, 0].set_ylabel("PDF")
    ax[3, 0].set_ylabel("PDF")
    ax[2, 0].hist(sel["mB_true"], bins=nbins, color="teal", alpha=0.3)
    ax[2, 0].set_xlabel(r"True $m_B$")
    ax[2, 0].set_ylabel("Number of SNe")
    ax[2, 1].hist(sel["z"], bins=nbins, color="teal", alpha=0.3)
    ax[2, 1].set_xlabel(r"$z$")
    for a in (ax[2, 2], ax[3, 1], ax[3, 2]):
        fig.delaxes(a)
    fig.suptitle(f"All DDFs, {len(sel)} SNe, mode: {mode}")
    fig.tight_layout()
    save_fig(fig, f"pulls_{mode}")

## Example light curve: truth, posterior and data

In [ ]:
def plot_example(sn_name, mode):
    r = fit_results[(mode, sn_name)]
    free_z = r["free_z"]
    source = sources[field_of[sn_name]]
    base = params_store[sn_name]
    pts = pts_by_sn[sn_name]
    t0_true = base["source.t0"]
    z_true = base["source.redshift"]
    tgrid = np.linspace(t0_true - 40 * (1 + z_true), t0_true + 60 * (1 + z_true), 400)
    colors = dict(zip(BANDS, ["C0", "C1", "C2", "C3", "C4", "C5"]))

    fig, ax = plt.subplots(figsize=(10, 5))
    for b in BANDS:
        pb_pts = pts[pts["filter"] == b]
        if len(pb_pts) == 0:
            continue
        fb = np.full(len(tgrid), b)
        true_flux = source.evaluate_bandfluxes(passband_group, tgrid, fb, GraphState.from_dict(base))
        fit_flux = source.evaluate_bandfluxes(
            passband_group, tgrid, fb, GraphState.from_dict(theta_to_params(base, r["theta_map"], free_z))
        )
        ax.errorbar(
            pb_pts["time"] - t0_true,
            pb_pts["flux"],
            yerr=pb_pts["fluxerr"],
            fmt="o",
            ms=3,
            c=colors[b],
            label=b,
        )
        ax.plot(tgrid - t0_true, true_flux, ls="dashed", c=colors[b], lw=1)
        ax.plot(tgrid - t0_true, fit_flux, c=colors[b], lw=1)
    ax.plot([], [], c="k", ls="dashed", label="True SALT3 model")
    ax.plot([], [], c="k", label="Fitted (max posterior)")
    ax.text(
        0.02,
        0.97,
        rf"$z^{{true}}$={z_true:.3f}, $x_1^{{true}}$={base['source.x1']:.2f}, $c^{{true}}$={base['source.c']:.3f}",
        transform=ax.transAxes,
        va="top",
    )
    names = dict(zip(r["names"], zip(r["mean"], r["std"])))
    ax.text(
        0.02,
        0.90,
        rf"$x_1^{{fit}}$={names['x1'][0]:.2f}$\pm${names['x1'][1]:.2f}, $c^{{fit}}$={names['c'][0]:.3f}$\pm${names['c'][1]:.3f}",
        transform=ax.transAxes,
        va="top",
    )
    ax.set_ylim(top=ax.get_ylim()[1] * 1.35)
    ax.legend(loc="upper right", ncol=2, frameon=False)
    ax.set_xlabel("Observer-frame phase (days)")
    ax.set_ylabel("Flux (nJy)")
    ax.set_title(f"{sn_name} ({mode})")
    return fig


sel = selected_tables[mode_main]
if len(sel):
    sn_example = sel["sn"].sample(n=1, random_state=SEED).iloc[0]
    fig = plot_example(sn_example, mode_main)
    save_fig(fig, f"example_lc_{sn_example}")